<a href="https://colab.research.google.com/github/WeiDeHuang1019/Fire-Detection/blob/main/fire_detecion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 稀疏光流

In [ ]:
import cv2
import numpy as np

# =========================
# Input / Output
# =========================
video_path = "fire2.mp4"

out_fire_path = "out_fire_sparse.mp4"
out_hsv_path = "out_hsv_mask.mp4"
out_flow_path = "out_sparse_flow.mp4"

cap = cv2.VideoCapture(video_path)

ret, prev_frame = cap.read()
if not ret:
    raise RuntimeError("Cannot read input video.")

fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out_fire = cv2.VideoWriter(out_fire_path, fourcc, fps, (w, h))
out_hsv = cv2.VideoWriter(out_hsv_path, fourcc, fps, (w, h))
out_flow = cv2.VideoWriter(out_flow_path, fourcc, fps, (w, h))

prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

# =========================
# Parameters
# =========================

# HSV fire-like region
lower_fire_color = np.array([0, 50, 120])
upper_fire_color = np.array([45, 255, 255])

# white / bright flame core
lower_fire_white = np.array([0, 0, 210])
upper_fire_white = np.array([179, 90, 255])

kernel = np.ones((5, 5), np.uint8)

# Sparse optical flow feature params
feature_params = dict(
    maxCorners=400,
    qualityLevel=0.01,
    minDistance=5,
    blockSize=7
)

lk_params = dict(
    winSize=(15, 15),
    maxLevel=2,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
)

# fire decision thresholds
min_area = 500
min_points = 15
mean_mag_threshold = 1.0
angle_var_threshold = 0.8

frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    result_frame = frame.copy()
    flow_vis = np.zeros_like(frame)

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # =========================
    # 1. HSV mask
    # =========================
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    mask_color = cv2.inRange(hsv, lower_fire_color, upper_fire_color)
    mask_white = cv2.inRange(hsv, lower_fire_white, upper_fire_white)

    # white core must be near red/yellow region
    mask_color_dilate = cv2.dilate(mask_color, kernel, iterations=2)
    mask_white_near_fire = cv2.bitwise_and(mask_white, mask_color_dilate)

    fire_mask = cv2.bitwise_or(mask_color, mask_white_near_fire)

    fire_mask = cv2.morphologyEx(fire_mask, cv2.MORPH_OPEN, kernel)
    fire_mask = cv2.morphologyEx(fire_mask, cv2.MORPH_CLOSE, kernel)

    hsv_mask_vis = cv2.cvtColor(fire_mask, cv2.COLOR_GRAY2BGR)

    # =========================
    # 2. Find contours from HSV mask
    # =========================
    contours, _ = cv2.findContours(
        fire_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue

        x, y, bw, bh = cv2.boundingRect(cnt)

        roi_mask = np.zeros_like(fire_mask)
        cv2.drawContours(roi_mask, [cnt], -1, 255, -1)

        # =========================
        # 3. Sparse feature points inside HSV candidate region
        # =========================
        p0 = cv2.goodFeaturesToTrack(
            prev_gray,
            mask=roi_mask,
            **feature_params
        )

        if p0 is None:
            continue

        p1, st, err = cv2.calcOpticalFlowPyrLK(
            prev_gray,
            gray,
            p0,
            None,
            **lk_params
        )

        if p1 is None or st is None:
            continue

        good_new = p1[st == 1]
        good_old = p0[st == 1]

        if len(good_new) == 0:
            continue

        dx = good_new[:, 0] - good_old[:, 0]
        dy = good_new[:, 1] - good_old[:, 1]

        mag = np.sqrt(dx ** 2 + dy ** 2)
        angle = np.arctan2(dy, dx)

        num_points = len(good_new)
        mean_mag = np.mean(mag)
        angle_var = np.var(angle)

        # =========================
        # 4. Draw sparse optical flow
        # =========================
        for new, old in zip(good_new, good_old):
            x_new, y_new = new.ravel()
            x_old, y_old = old.ravel()

            cv2.circle(flow_vis, (int(x_new), int(y_new)), 3, (0, 255, 255), -1)
            cv2.line(
                flow_vis,
                (int(x_old), int(y_old)),
                (int(x_new), int(y_new)),
                (0, 255, 0),
                1
            )

            cv2.circle(result_frame, (int(x_new), int(y_new)), 2, (0, 255, 255), -1)

        # =========================
        # 5. Fire decision
        # =========================
        is_fire = (
            num_points >= min_points and
            mean_mag >= mean_mag_threshold and
            angle_var >= angle_var_threshold
        )

        if is_fire:
            cv2.rectangle(result_frame, (x, y), (x + bw, y + bh), (0, 0, 255), 2)
            cv2.putText(
                result_frame,
                f"FIRE pts={num_points} mag={mean_mag:.2f}",
                (x, max(y - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255),
                2
            )

    # =========================
    # 6. Write output videos
    # =========================
    out_fire.write(result_frame)
    out_hsv.write(hsv_mask_vis)
    out_flow.write(flow_vis)

    prev_gray = gray.copy()
    frame_idx += 1

cap.release()
out_fire.release()
out_hsv.release()
out_flow.release()

print("Done.")
print("Fire result:", out_fire_path)
print("HSV mask:", out_hsv_path)
print("Sparse flow:", out_flow_path)

Done.
Fire result: out_fire_sparse.mp4
HSV mask: out_hsv_mask.mp4
Sparse flow: out_sparse_flow.mp4


# HSV抓ROI再對ROI跑密集光流

In [ ]:
import time
import cv2
import numpy as np
from base64 import b64encode
from IPython.display import HTML

# ============================================================
# 1. Input / Output Settings
# ============================================================

video_path = "fire.mp4"

out_fire_path = "/content/out_fire_detection.mp4"
out_hsv_path = "/content/out_hsv_mask.mp4"
out_flow_path = "/content/out_dense_flow_mask.mp4"

cap = cv2.VideoCapture(video_path)

ret, prev_frame = cap.read()
if not ret:
    raise RuntimeError("Cannot read input video. Please check video_path.")

fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if fps == 0 or np.isnan(fps):
    fps = 30

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out_fire = cv2.VideoWriter(out_fire_path, fourcc, fps, (w, h))
out_hsv = cv2.VideoWriter(out_hsv_path, fourcc, fps, (w, h))
out_flow = cv2.VideoWriter(out_flow_path, fourcc, fps, (w, h))

prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

# ============================================================
# 2. Parameters
# ============================================================

# ---------- HSV fire candidate parameters ----------
# 紅 / 橘 / 黃火焰區域
lower_fire_color = np.array([0, 50, 120])
upper_fire_color = np.array([45, 255, 255])

# 白亮火焰核心：針對火焰偏白、過曝的情況
lower_white_fire = np.array([0, 0, 230])
upper_white_fire = np.array([179, 80, 255])

# 淡黃色高亮區域
lower_pale_yellow = np.array([10, 20, 200])
upper_pale_yellow = np.array([45, 120, 255])

# ---------- Morphology ----------
kernel = np.ones((5, 5), np.uint8)

# ---------- Contour filtering ----------
min_area = 300        # 候選區最小面積
max_area_ratio = 0.7  # 避免整張畫面被判成候選區

# ---------- Dense optical flow parameters ----------
flow_mag_threshold = 1.2        # 單一 pixel 光流強度門檻
mean_mag_threshold = 0.6        # ROI 平均光流強度門檻
motion_density_threshold = 0.12 # ROI 內高光流比例門檻

# ---------- Fire decision ----------
min_fire_score = 2

# ============================================================
# 3. Helper Functions
# ============================================================

def build_hsv_fire_mask(frame):
    """
    建立 HSV 火焰候選遮罩。
    包含：
    1. 紅橘黃色高飽和區域
    2. 白亮核心區域
    3. 淡黃色高亮區域
    """

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # 紅 / 橘 / 黃外焰
    mask_color = cv2.inRange(hsv, lower_fire_color, upper_fire_color)

    # 白亮火焰核心
    mask_white = cv2.inRange(hsv, lower_white_fire, upper_white_fire)

    # 淡黃色高亮火焰
    mask_pale_yellow = cv2.inRange(hsv, lower_pale_yellow, upper_pale_yellow)

    # 合併 HSV 候選區
    fire_mask = cv2.bitwise_or(mask_color, mask_white)
    fire_mask = cv2.bitwise_or(fire_mask, mask_pale_yellow)

    # 去雜訊
    fire_mask = cv2.morphologyEx(fire_mask, cv2.MORPH_OPEN, kernel)
    fire_mask = cv2.morphologyEx(fire_mask, cv2.MORPH_CLOSE, kernel)

    return fire_mask


def compute_dense_flow(prev_gray, gray):
    """
    使用 Farneback Dense Optical Flow 計算每個 pixel 的運動向量。
    回傳：
    - mag: 光流向量長度
    - ang: 光流方向
    """

    flow = cv2.calcOpticalFlowFarneback(
        prev_gray,
        gray,
        None,
        pyr_scale=0.5,
        levels=3,
        winsize=15,
        iterations=3,
        poly_n=5,
        poly_sigma=1.2,
        flags=0
    )

    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

    return mag, ang


def build_flow_mask(mag):
    """
    將 dense optical flow magnitude 轉成二值遮罩。
    """

    flow_mask = (mag > flow_mag_threshold).astype(np.uint8) * 255
    flow_mask = cv2.morphologyEx(flow_mask, cv2.MORPH_OPEN, kernel)

    return flow_mask


def evaluate_fire_region(roi_mask, roi_mag):
    """
    針對 HSV 候選區 ROI 評估是否符合火焰動態特徵。
    """

    valid_pixels = roi_mask > 0

    if np.sum(valid_pixels) == 0:
        return False, 0.0, 0.0, 0

    roi_motion = roi_mag[valid_pixels]

    mean_mag = float(np.mean(roi_motion))
    motion_density = float(np.sum(roi_motion > flow_mag_threshold) / len(roi_motion))

    score = 0

    if mean_mag >= mean_mag_threshold:
        score += 1

    if motion_density >= motion_density_threshold:
        score += 1

    is_fire = score >= min_fire_score

    return is_fire, mean_mag, motion_density, score


def show_video(path, width=700):
    """
    Colab 顯示 mp4 影片用。
    """
    mp4 = open(path, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

    return HTML(f"""
    <video width="{width}" controls>
        <source src="{data_url}" type="video/mp4">
    </video>
    """)


# ============================================================
# 4. Main Processing Loop
# ============================================================

frame_idx = 0

start_time = time.time()
processed_frames = 0
fps_list = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    result_frame = frame.copy()

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # ------------------------------------------------------------
    # Step 1: HSV candidate mask
    # ------------------------------------------------------------
    fire_mask = build_hsv_fire_mask(frame)

    hsv_vis = cv2.cvtColor(fire_mask, cv2.COLOR_GRAY2BGR)

    # ------------------------------------------------------------
    # Step 2: Dense Optical Flow
    # ------------------------------------------------------------
    mag, ang = compute_dense_flow(prev_gray, gray)

    flow_mask = build_flow_mask(mag)
    flow_vis = cv2.cvtColor(flow_mask, cv2.COLOR_GRAY2BGR)

    # 只保留 HSV 疑似火焰區域內的光流
    flow_in_hsv = cv2.bitwise_and(flow_mask, fire_mask)
    flow_in_hsv_vis = cv2.cvtColor(flow_in_hsv, cv2.COLOR_GRAY2BGR)

    # ------------------------------------------------------------
    # Step 3: Contour analysis on HSV mask
    # ------------------------------------------------------------
    contours, _ = cv2.findContours(
        fire_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if area < min_area:
            continue

        if area > max_area_ratio * w * h:
            continue

        x, y, bw, bh = cv2.boundingRect(cnt)

        roi_mask = fire_mask[y:y+bh, x:x+bw]
        roi_mag = mag[y:y+bh, x:x+bw]

        # ------------------------------------------------------------
        # Step 4: Fire decision by HSV + Dense Flow
        # ------------------------------------------------------------
        is_fire, mean_mag, motion_density, score = evaluate_fire_region(
            roi_mask,
            roi_mag
        )

        if is_fire:
            cv2.rectangle(
                result_frame,
                (x, y),
                (x + bw, y + bh),
                (0, 0, 255),
                2
            )

            label = f"FIRE M={mean_mag:.2f} D={motion_density:.2f}"

            cv2.putText(
                result_frame,
                label,
                (x, max(y - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (0, 0, 255),
                2
            )

            # 在 HSV mask 上也畫出火焰框，方便 debug
            cv2.rectangle(
                hsv_vis,
                (x, y),
                (x + bw, y + bh),
                (0, 0, 255),
                2
            )

            # 在 flow mask 上也畫出火焰框
            cv2.rectangle(
                flow_in_hsv_vis,
                (x, y),
                (x + bw, y + bh),
                (0, 0, 255),
                2
            )

    processed_frames += 1
    elapsed_time = time.time() - start_time
    processing_fps = processed_frames / elapsed_time
    fps_list.append(processing_fps)

    cv2.putText(
        result_frame,
        f"FPS: {processing_fps:.2f}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

    # ------------------------------------------------------------
    # Step 5: Write videos
    # ------------------------------------------------------------
    out_fire.write(result_frame)
    out_hsv.write(hsv_vis)
    out_flow.write(flow_in_hsv_vis)

    prev_gray = gray.copy()
    frame_idx += 1

cap.release()
out_fire.release()
out_hsv.release()
out_flow.release()

avg_fps = processed_frames / (time.time() - start_time)

print("Done.")
print(f"Processed frames: {processed_frames}")
print(f"Average Processing FPS: {avg_fps:.2f}")
print(f"Input Video FPS: {fps:.2f}")
print("Fire detection output:", out_fire_path)
print("HSV mask output:", out_hsv_path)
print("Dense flow mask output:", out_flow_path)

print("Done.")
print("Fire detection output:", out_fire_path)
print("HSV mask output:", out_hsv_path)
print("Dense flow mask output:", out_flow_path)

Done.
Processed frames: 320
Average Processing FPS: 2.74
Input Video FPS: 30.00
Fire detection output: /content/out_fire_detection.mp4
HSV mask output: /content/out_hsv_mask.mp4
Dense flow mask output: /content/out_dense_flow_mask.mp4
Done.
Fire detection output: /content/out_fire_detection.mp4
HSV mask output: /content/out_hsv_mask.mp4
Dense flow mask output: /content/out_dense_flow_mask.mp4


# HSV+密集光流 (降低解析度+farneback參數調低)

In [ ]:
import time
import cv2
import numpy as np
from base64 import b64encode
from IPython.display import HTML

# ============================================================
# 1. Input / Output Settings
# ============================================================

video_path = "fire.mp4"

out_fire_path = "/content/out_fire_detection_fast.mp4"
out_hsv_path = "/content/out_hsv_mask_fast.mp4"
out_flow_path = "/content/out_dense_flow_mask_fast.mp4"

cap = cv2.VideoCapture(video_path)

ret, prev_frame_full = cap.read()
if not ret:
    raise RuntimeError("Cannot read input video. Please check video_path.")

fps = cap.get(cv2.CAP_PROP_FPS)
orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if fps == 0 or np.isnan(fps):
    fps = 30

# ============================================================
# 1.1 Processing Resolution
# ============================================================
# 這裡是主要加速點：演算法不在原尺寸上跑，而是在小圖上跑
process_w = 640
process_h = 480

scale_x = orig_w / process_w
scale_y = orig_h / process_h

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

# 輸出仍然維持原影片尺寸，方便觀看與報告
out_fire = cv2.VideoWriter(out_fire_path, fourcc, fps, (orig_w, orig_h))
out_hsv = cv2.VideoWriter(out_hsv_path, fourcc, fps, (orig_w, orig_h))
out_flow = cv2.VideoWriter(out_flow_path, fourcc, fps, (orig_w, orig_h))

# 第一張 frame 也先 resize，再轉灰階
prev_frame_small = cv2.resize(prev_frame_full, (process_w, process_h))
prev_gray = cv2.cvtColor(prev_frame_small, cv2.COLOR_BGR2GRAY)

# ============================================================
# 2. Parameters
# ============================================================

# ---------- HSV fire candidate parameters ----------
# 紅 / 橘 / 黃火焰區域
lower_fire_color = np.array([0, 50, 120])
upper_fire_color = np.array([45, 255, 255])

# 白亮火焰核心：針對火焰偏白、過曝的情況
lower_white_fire = np.array([0, 0, 230])
upper_white_fire = np.array([179, 80, 255])

# 淡黃色高亮區域
lower_pale_yellow = np.array([10, 20, 200])
upper_pale_yellow = np.array([45, 120, 255])

# ---------- Morphology ----------
kernel = np.ones((5, 5), np.uint8)

# ---------- Contour filtering ----------
# 原本 min_area = 300 是針對原尺寸，現在要換算到小尺寸
min_area_original = 300
min_area = max(20, int(min_area_original / (scale_x * scale_y)))

max_area_ratio = 0.7  # 避免整張畫面被判成候選區

# ---------- Dense optical flow parameters ----------
flow_mag_threshold = 1.2
mean_mag_threshold = 0.6
motion_density_threshold = 0.12

# ---------- Fire decision ----------
min_fire_score = 2

# ============================================================
# 3. Helper Functions
# ============================================================

def build_hsv_fire_mask(frame_small):
    """
    建立 HSV 火焰候選遮罩。
    注意：這裡輸入的是 resize 後的小圖。
    """

    hsv = cv2.cvtColor(frame_small, cv2.COLOR_BGR2HSV)

    # 紅 / 橘 / 黃外焰
    mask_color = cv2.inRange(hsv, lower_fire_color, upper_fire_color)

    # 白亮火焰核心
    mask_white = cv2.inRange(hsv, lower_white_fire, upper_white_fire)

    # 淡黃色高亮火焰
    mask_pale_yellow = cv2.inRange(hsv, lower_pale_yellow, upper_pale_yellow)

    # 合併 HSV 候選區
    fire_mask = cv2.bitwise_or(mask_color, mask_white)
    fire_mask = cv2.bitwise_or(fire_mask, mask_pale_yellow)

    # 去雜訊
    fire_mask = cv2.morphologyEx(fire_mask, cv2.MORPH_OPEN, kernel)
    fire_mask = cv2.morphologyEx(fire_mask, cv2.MORPH_CLOSE, kernel)

    return fire_mask


def compute_dense_flow_fast(prev_gray, gray):
    """
    較輕量的 Farneback Dense Optical Flow。
    這裡是主要加速點：
    - levels: 3 -> 1
    - winsize: 15 -> 9
    - iterations: 3 -> 1
    """

    flow = cv2.calcOpticalFlowFarneback(
        prev_gray,
        gray,
        None,
        pyr_scale=0.5,
        levels=1,
        winsize=9,
        iterations=1,
        poly_n=5,
        poly_sigma=1.2,
        flags=0
    )

    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

    return mag, ang


def build_flow_mask(mag):
    """
    將 dense optical flow magnitude 轉成二值遮罩。
    """

    flow_mask = (mag > flow_mag_threshold).astype(np.uint8) * 255
    flow_mask = cv2.morphologyEx(flow_mask, cv2.MORPH_OPEN, kernel)

    return flow_mask


def evaluate_fire_region(roi_mask, roi_mag):
    """
    針對 HSV 候選區 ROI 評估是否符合火焰動態特徵。
    """

    valid_pixels = roi_mask > 0

    if np.sum(valid_pixels) == 0:
        return False, 0.0, 0.0, 0

    roi_motion = roi_mag[valid_pixels]

    mean_mag = float(np.mean(roi_motion))
    motion_density = float(np.sum(roi_motion > flow_mag_threshold) / len(roi_motion))

    score = 0

    if mean_mag >= mean_mag_threshold:
        score += 1

    if motion_density >= motion_density_threshold:
        score += 1

    is_fire = score >= min_fire_score

    return is_fire, mean_mag, motion_density, score


def show_video(path, width=700):
    """
    Colab 顯示 mp4 影片用。
    """
    mp4 = open(path, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

    return HTML(f"""
    <video width="{width}" controls>
        <source src="{data_url}" type="video/mp4">
    </video>
    """)


# ============================================================
# 4. Main Processing Loop
# ============================================================

frame_idx = 0

start_time = time.time()
processed_frames = 0
fps_list = []

while True:
    ret, frame_full = cap.read()
    if not ret:
        break

    result_frame = frame_full.copy()

    # ------------------------------------------------------------
    # Step 0: Resize frame for faster processing
    # ------------------------------------------------------------
    frame_small = cv2.resize(frame_full, (process_w, process_h))
    gray = cv2.cvtColor(frame_small, cv2.COLOR_BGR2GRAY)

    # ------------------------------------------------------------
    # Step 1: HSV candidate mask on small frame
    # ------------------------------------------------------------
    fire_mask = build_hsv_fire_mask(frame_small)

    hsv_vis_small = cv2.cvtColor(fire_mask, cv2.COLOR_GRAY2BGR)

    # ------------------------------------------------------------
    # Step 2: Fast Dense Optical Flow on small frame
    # ------------------------------------------------------------
    mag, ang = compute_dense_flow_fast(prev_gray, gray)

    flow_mask = build_flow_mask(mag)

    # 只保留 HSV 疑似火焰區域內的光流
    flow_in_hsv = cv2.bitwise_and(flow_mask, fire_mask)
    flow_in_hsv_vis_small = cv2.cvtColor(flow_in_hsv, cv2.COLOR_GRAY2BGR)

    # ------------------------------------------------------------
    # Step 3: Contour analysis on HSV mask
    # ------------------------------------------------------------
    contours, _ = cv2.findContours(
        fire_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if area < min_area:
            continue

        if area > max_area_ratio * process_w * process_h:
            continue

        x, y, bw, bh = cv2.boundingRect(cnt)

        roi_mask = fire_mask[y:y+bh, x:x+bw]
        roi_mag = mag[y:y+bh, x:x+bw]

        # ------------------------------------------------------------
        # Step 4: Fire decision by HSV + Dense Flow
        # ------------------------------------------------------------
        is_fire, mean_mag, motion_density, score = evaluate_fire_region(
            roi_mask,
            roi_mag
        )

        if is_fire:
            # 將小圖座標 scale 回原圖座標
            X1 = int(x * scale_x)
            Y1 = int(y * scale_y)
            X2 = int((x + bw) * scale_x)
            Y2 = int((y + bh) * scale_y)

            cv2.rectangle(
                result_frame,
                (X1, Y1),
                (X2, Y2),
                (0, 0, 255),
                2
            )

            label = f"FIRE M={mean_mag:.2f} D={motion_density:.2f}"

            cv2.putText(
                result_frame,
                label,
                (X1, max(Y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (0, 0, 255),
                2
            )

            # 在小尺寸 HSV mask 上也畫出火焰框
            cv2.rectangle(
                hsv_vis_small,
                (x, y),
                (x + bw, y + bh),
                (0, 0, 255),
                1
            )

            # 在小尺寸 flow mask 上也畫出火焰框
            cv2.rectangle(
                flow_in_hsv_vis_small,
                (x, y),
                (x + bw, y + bh),
                (0, 0, 255),
                1
            )

    processed_frames += 1
    elapsed_time = time.time() - start_time
    processing_fps = processed_frames / elapsed_time
    fps_list.append(processing_fps)

    cv2.putText(
        result_frame,
        f"FPS: {processing_fps:.2f}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

    # ------------------------------------------------------------
    # Step 5: Resize debug videos back to original size
    # ------------------------------------------------------------
    hsv_vis_full = cv2.resize(hsv_vis_small, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
    flow_vis_full = cv2.resize(flow_in_hsv_vis_small, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)

    # ------------------------------------------------------------
    # Step 6: Write videos
    # ------------------------------------------------------------
    out_fire.write(result_frame)
    out_hsv.write(hsv_vis_full)
    out_flow.write(flow_vis_full)

    prev_gray = gray.copy()
    frame_idx += 1

cap.release()
out_fire.release()
out_hsv.release()
out_flow.release()

avg_fps = processed_frames / (time.time() - start_time)

print("Done.")
print(f"Original resolution: {orig_w} x {orig_h}")
print(f"Processing resolution: {process_w} x {process_h}")
print(f"Processed frames: {processed_frames}")
print(f"Average Processing FPS: {avg_fps:.2f}")
print(f"Input Video FPS: {fps:.2f}")
print("Fire detection output:", out_fire_path)
print("HSV mask output:", out_hsv_path)
print("Dense flow mask output:", out_flow_path)

Done.
Original resolution: 1280 x 720
Processing resolution: 640 x 480
Processed frames: 320
Average Processing FPS: 11.12
Input Video FPS: 30.00
Fire detection output: /content/out_fire_detection_fast.mp4
HSV mask output: /content/out_hsv_mask_fast.mp4
Dense flow mask output: /content/out_dense_flow_mask_fast.mp4


# HSV -> ROI -> farneback

In [ ]:
import time
import cv2
import numpy as np
from base64 import b64encode
from IPython.display import HTML

# ============================================================
# 1. Input / Output Settings
# ============================================================

video_path = "fire.mp4"

out_fire_path = "/content/out_fire_detection_roi_flow.mp4"
out_hsv_path = "/content/out_hsv_mask_roi_flow.mp4"
out_flow_path = "/content/out_dense_flow_mask_roi_flow.mp4"

cap = cv2.VideoCapture(video_path)

ret, prev_frame_full = cap.read()
if not ret:
    raise RuntimeError("Cannot read input video. Please check video_path.")

fps = cap.get(cv2.CAP_PROP_FPS)
orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if fps == 0 or np.isnan(fps):
    fps = 30

# ============================================================
# 1.1 Processing Resolution
# ============================================================

# 演算法處理解析度，越小越快
process_w = 640
process_h = 480

scale_x = orig_w / process_w
scale_y = orig_h / process_h

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

# 輸出仍維持原圖尺寸
out_fire = cv2.VideoWriter(out_fire_path, fourcc, fps, (orig_w, orig_h))
out_hsv = cv2.VideoWriter(out_hsv_path, fourcc, fps, (orig_w, orig_h))
out_flow = cv2.VideoWriter(out_flow_path, fourcc, fps, (orig_w, orig_h))

# previous frame 也縮小後再轉灰階
prev_frame_small = cv2.resize(prev_frame_full, (process_w, process_h))
prev_gray = cv2.cvtColor(prev_frame_small, cv2.COLOR_BGR2GRAY)

# ============================================================
# 2. Parameters
# ============================================================

# ---------- HSV fire candidate parameters ----------

# 紅 / 橘 / 黃火焰區域
lower_fire_color = np.array([0, 0, 200])
upper_fire_color = np.array([90, 255, 255])

# 白亮火焰核心：針對偏白、過曝火焰
lower_white_fire = np.array([0, 0, 230])
upper_white_fire = np.array([179, 80, 255])

# 淡黃色高亮區域
lower_pale_yellow = np.array([10, 20, 200])
upper_pale_yellow = np.array([45, 120, 255])

# ---------- Morphology ----------
kernel = np.ones((5, 5), np.uint8)

# ---------- Contour filtering ----------
# 原本 min_area = 300 是原尺寸概念，縮小後要換算
min_area_original = 150
min_area = max(20, int(min_area_original / (scale_x * scale_y)))

max_area_ratio = 0.7

# ROI padding，避免 bbox 太緊導致光流不穩
roi_pad = 5

# 太小的 ROI 不跑 optical flow
min_roi_w = 10
min_roi_h = 10

# ---------- Dense optical flow parameters ----------
flow_mag_threshold = 1.2
mean_mag_threshold = 0.6
motion_density_threshold = 0.12

# ---------- Fire decision ----------
min_fire_score = 2

# ============================================================
# 3. Helper Functions
# ============================================================

def build_hsv_fire_mask(frame_small):
    """
    建立 HSV 火焰候選遮罩。
    輸入為縮小後的小圖。
    """

    hsv = cv2.cvtColor(frame_small, cv2.COLOR_BGR2HSV)

    # 紅 / 橘 / 黃外焰
    mask_color = cv2.inRange(hsv, lower_fire_color, upper_fire_color)

    # 白亮火焰核心
    mask_white = cv2.inRange(hsv, lower_white_fire, upper_white_fire)

    # 淡黃色高亮火焰
    mask_pale_yellow = cv2.inRange(hsv, lower_pale_yellow, upper_pale_yellow)

    # 合併 HSV 候選區
    #fire_mask = cv2.bitwise_or(mask_color, mask_white)
    #fire_mask = cv2.bitwise_or(fire_mask, mask_pale_yellow)
    fire_mask = mask_color

    # 去雜訊
    fire_mask = cv2.morphologyEx(fire_mask, cv2.MORPH_OPEN, kernel)
    fire_mask = cv2.morphologyEx(fire_mask, cv2.MORPH_CLOSE, kernel)

    return fire_mask


def compute_dense_flow_roi(prev_gray_roi, gray_roi):
    """
    只針對 ROI 計算 Farneback Dense Optical Flow。
    這才是真正 ROI-based Dense Flow。
    """

    flow = cv2.calcOpticalFlowFarneback(
        prev_gray_roi,
        gray_roi,
        None,
        pyr_scale=0.5,
        levels=1,
        winsize=9,
        iterations=1,
        poly_n=5,
        poly_sigma=1.1,
        flags=0
    )

    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

    return mag, ang


def evaluate_fire_region(roi_mask, roi_mag):
    """
    針對 HSV 候選 ROI 評估是否符合火焰動態特徵。
    """

    valid_pixels = roi_mask > 0

    if np.sum(valid_pixels) == 0:
        return False, 0.0, 0.0, 0

    roi_motion = roi_mag[valid_pixels]

    mean_mag = float(np.mean(roi_motion))
    motion_density = float(np.sum(roi_motion > flow_mag_threshold) / len(roi_motion))

    score = 0

    if mean_mag >= mean_mag_threshold:
        score += 1

    if motion_density >= motion_density_threshold:
        score += 1

    is_fire = score >= min_fire_score

    return is_fire, mean_mag, motion_density, score


def show_video(path, width=700):
    """
    Colab 顯示 mp4 影片用。
    """

    mp4 = open(path, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

    return HTML(f"""
    <video width="{width}" controls>
        <source src="{data_url}" type="video/mp4">
    </video>
    """)


# ============================================================
# 4. Main Processing Loop
# ============================================================

frame_idx = 0

start_time = time.time()
processed_frames = 0
fps_list = []

while True:
    ret, frame_full = cap.read()
    if not ret:
        break

    result_frame = frame_full.copy()

    # ------------------------------------------------------------
    # Step 0: Resize frame for faster processing
    # ------------------------------------------------------------

    frame_small = cv2.resize(frame_full, (process_w, process_h))
    gray = cv2.cvtColor(frame_small, cv2.COLOR_BGR2GRAY)

    # ------------------------------------------------------------
    # Step 1: HSV candidate mask on small frame
    # ------------------------------------------------------------

    fire_mask = build_hsv_fire_mask(frame_small)

    hsv_vis_small = cv2.cvtColor(fire_mask, cv2.COLOR_GRAY2BGR)

    # flow debug canvas：只有 ROI 內會被填入光流結果
    flow_debug_small = np.zeros((process_h, process_w), dtype=np.uint8)

    # ------------------------------------------------------------
    # Step 2: Find HSV candidate contours first
    # ------------------------------------------------------------

    contours, _ = cv2.findContours(
        fire_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # ------------------------------------------------------------
    # Step 3: Run Dense Optical Flow only inside each ROI
    # ------------------------------------------------------------

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if area < min_area:
            continue

        if area > max_area_ratio * process_w * process_h:
            continue

        x, y, bw, bh = cv2.boundingRect(cnt)

        # 加 padding，避免 ROI 太緊
        x1 = max(x - roi_pad, 0)
        y1 = max(y - roi_pad, 0)
        x2 = min(x + bw + roi_pad, process_w)
        y2 = min(y + bh + roi_pad, process_h)

        roi_w = x2 - x1
        roi_h = y2 - y1

        if roi_w < min_roi_w or roi_h < min_roi_h:
            continue

        # 只切 ROI
        prev_roi = prev_gray[y1:y2, x1:x2]
        gray_roi = gray[y1:y2, x1:x2]
        roi_mask = fire_mask[y1:y2, x1:x2]

        # ------------------------------------------------------------
        # 真正 ROI-based Dense Optical Flow
        # ------------------------------------------------------------

        roi_mag, roi_ang = compute_dense_flow_roi(prev_roi, gray_roi)

        # ------------------------------------------------------------
        # Fire decision by HSV ROI + ROI Dense Flow
        # ------------------------------------------------------------

        is_fire, mean_mag, motion_density, score = evaluate_fire_region(
            roi_mask,
            roi_mag
        )

        # ------------------------------------------------------------
        # Build ROI flow debug mask
        # ------------------------------------------------------------

        roi_flow_mask = (roi_mag > flow_mag_threshold).astype(np.uint8) * 255
        roi_flow_mask = cv2.morphologyEx(roi_flow_mask, cv2.MORPH_OPEN, kernel)

        # 將 ROI flow mask 填回整張小圖的 flow debug canvas
        flow_debug_small[y1:y2, x1:x2] = cv2.bitwise_or(
            flow_debug_small[y1:y2, x1:x2],
            roi_flow_mask
        )

        # ------------------------------------------------------------
        # Draw result if fire
        # ------------------------------------------------------------

        if is_fire:
            # 將小圖座標 scale 回原圖座標
            X1 = int(x1 * scale_x)
            Y1 = int(y1 * scale_y)
            X2 = int(x2 * scale_x)
            Y2 = int(y2 * scale_y)

            cv2.rectangle(
                result_frame,
                (X1, Y1),
                (X2, Y2),
                (0, 0, 255),
                2
            )

            label = f"FIRE M={mean_mag:.2f} D={motion_density:.2f}"

            cv2.putText(
                result_frame,
                label,
                (X1, max(Y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (0, 0, 255),
                2
            )

            # 在 HSV mask 上畫出火焰框
            cv2.rectangle(
                hsv_vis_small,
                (x1, y1),
                (x2, y2),
                (0, 0, 255),
                1
            )

            # 在 flow debug 上畫出火焰框
            cv2.rectangle(
                flow_debug_small,
                (x1, y1),
                (x2, y2),
                255,
                1
            )

    # ------------------------------------------------------------
    # Step 4: FPS calculation
    # ------------------------------------------------------------

    processed_frames += 1
    elapsed_time = time.time() - start_time
    processing_fps = processed_frames / elapsed_time
    fps_list.append(processing_fps)

    cv2.putText(
        result_frame,
        f"FPS: {processing_fps:.2f}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

    # ------------------------------------------------------------
    # Step 5: Resize debug videos back to original size
    # ------------------------------------------------------------

    flow_in_hsv_vis_small = cv2.cvtColor(flow_debug_small, cv2.COLOR_GRAY2BGR)

    hsv_vis_full = cv2.resize(
        hsv_vis_small,
        (orig_w, orig_h),
        interpolation=cv2.INTER_NEAREST
    )

    flow_vis_full = cv2.resize(
        flow_in_hsv_vis_small,
        (orig_w, orig_h),
        interpolation=cv2.INTER_NEAREST
    )

    # ------------------------------------------------------------
    # Step 6: Write videos
    # ------------------------------------------------------------

    out_fire.write(result_frame)
    out_hsv.write(hsv_vis_full)
    out_flow.write(flow_vis_full)

    # 更新 previous frame
    prev_gray = gray.copy()
    frame_idx += 1

cap.release()
out_fire.release()
out_hsv.release()
out_flow.release()

avg_fps = processed_frames / (time.time() - start_time)

print("Done.")
print(f"Original resolution: {orig_w} x {orig_h}")
print(f"Processing resolution: {process_w} x {process_h}")
print(f"Processed frames: {processed_frames}")
print(f"Average Processing FPS: {avg_fps:.2f}")
print(f"Input Video FPS: {fps:.2f}")
print("Fire detection output:", out_fire_path)
print("HSV mask output:", out_hsv_path)
print("Dense flow mask output:", out_flow_path)

Done.
Original resolution: 1280 x 720
Processing resolution: 640 x 480
Processed frames: 320
Average Processing FPS: 35.88
Input Video FPS: 30.00
Fire detection output: /content/out_fire_detection_roi_flow.mp4
HSV mask output: /content/out_hsv_mask_roi_flow.mp4
Dense flow mask output: /content/out_dense_flow_mask_roi_flow.mp4


# HSV -> ROI -> farneback (Frame skipping)

In [ ]:
import time
import cv2
import numpy as np
from base64 import b64encode
from IPython.display import HTML

# ============================================================
# 1. Input / Output Settings
# ============================================================

video_path = "fire.mp4"

out_fire_path = "/content/out_fire_detection_roi_flow_skip.mp4"
out_hsv_path = "/content/out_hsv_mask_roi_flow_skip.mp4"
out_flow_path = "/content/out_dense_flow_mask_roi_flow_skip.mp4"

cap = cv2.VideoCapture(video_path)

ret, prev_frame_full = cap.read()
if not ret:
    raise RuntimeError("Cannot read input video. Please check video_path.")

fps = cap.get(cv2.CAP_PROP_FPS)
orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if fps == 0 or np.isnan(fps):
    fps = 30

# ============================================================
# 1.1 Processing Resolution
# ============================================================

process_w = 640
process_h = 480

scale_x = orig_w / process_w
scale_y = orig_h / process_h

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out_fire = cv2.VideoWriter(out_fire_path, fourcc, fps, (orig_w, orig_h))
out_hsv = cv2.VideoWriter(out_hsv_path, fourcc, fps, (orig_w, orig_h))
out_flow = cv2.VideoWriter(out_flow_path, fourcc, fps, (orig_w, orig_h))

prev_frame_small = cv2.resize(prev_frame_full, (process_w, process_h))
prev_gray = cv2.cvtColor(prev_frame_small, cv2.COLOR_BGR2GRAY)

# ============================================================
# 2. Parameters
# ============================================================

# ---------- Detection interval ----------
# 每 N 幀才跑一次完整偵測，其餘幀沿用上一輪 bounding boxes
detection_interval = 2

# 儲存上一輪偵測結果
last_fire_boxes = []
last_hsv_vis_small = np.zeros((process_h, process_w, 3), dtype=np.uint8)
last_flow_vis_small = np.zeros((process_h, process_w, 3), dtype=np.uint8)

# ---------- HSV fire candidate parameters ----------

lower_fire_color = np.array([0, 0, 200])
upper_fire_color = np.array([90, 255, 255])

lower_white_fire = np.array([0, 0, 230])
upper_white_fire = np.array([179, 80, 255])

lower_pale_yellow = np.array([10, 20, 200])
upper_pale_yellow = np.array([45, 120, 255])

# ---------- Morphology ----------
kernel = np.ones((5, 5), np.uint8)

# ---------- Contour filtering ----------
min_area_original = 100
min_area = max(20, int(min_area_original / (scale_x * scale_y)))

max_area_ratio = 0.7

roi_pad = 5
min_roi_w = 10
min_roi_h = 10

# ---------- Dense optical flow parameters ----------
flow_mag_threshold = 1.2
mean_mag_threshold = 0.6
motion_density_threshold = 0.12

# ---------- Fire decision ----------
min_fire_score = 2

# ============================================================
# 3. Helper Functions
# ============================================================

def build_hsv_fire_mask(frame_small):
    hsv = cv2.cvtColor(frame_small, cv2.COLOR_BGR2HSV)

    mask_color = cv2.inRange(hsv, lower_fire_color, upper_fire_color)
    mask_white = cv2.inRange(hsv, lower_white_fire, upper_white_fire)
    mask_pale_yellow = cv2.inRange(hsv, lower_pale_yellow, upper_pale_yellow)

    #fire_mask = cv2.bitwise_or(mask_color, mask_white)
    #fire_mask = cv2.bitwise_or(fire_mask, mask_pale_yellow)
    fire_mask = mask_color

    fire_mask = cv2.morphologyEx(fire_mask, cv2.MORPH_OPEN, kernel)
    fire_mask = cv2.morphologyEx(fire_mask, cv2.MORPH_CLOSE, kernel)

    return fire_mask


def compute_dense_flow_roi(prev_gray_roi, gray_roi):
    flow = cv2.calcOpticalFlowFarneback(
        prev_gray_roi,
        gray_roi,
        None,
        pyr_scale=0.5,
        levels=1,
        winsize=9,
        iterations=1,
        poly_n=5,
        poly_sigma=1.1,
        flags=0
    )

    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    return mag, ang


def evaluate_fire_region(roi_mask, roi_mag):
    valid_pixels = roi_mask > 0

    if np.sum(valid_pixels) == 0:
        return False, 0.0, 0.0, 0

    roi_motion = roi_mag[valid_pixels]

    mean_mag = float(np.mean(roi_motion))
    motion_density = float(np.sum(roi_motion > flow_mag_threshold) / len(roi_motion))

    score = 0

    if mean_mag >= mean_mag_threshold:
        score += 1

    if motion_density >= motion_density_threshold:
        score += 1

    is_fire = score >= min_fire_score

    return is_fire, mean_mag, motion_density, score


def draw_last_boxes(result_frame, boxes):
    """
    在非偵測幀上沿用上一輪 bounding boxes。
    boxes 格式：
    [
        {
            "box": (X1, Y1, X2, Y2),
            "mean_mag": mean_mag,
            "motion_density": motion_density
        }
    ]
    """

    for item in boxes:
        X1, Y1, X2, Y2 = item["box"]
        mean_mag = item["mean_mag"]
        motion_density = item["motion_density"]

        cv2.rectangle(
            result_frame,
            (X1, Y1),
            (X2, Y2),
            (0, 0, 255),
            2
        )

        label = f"FIRE M={mean_mag:.2f} D={motion_density:.2f}"

        cv2.putText(
            result_frame,
            label,
            (X1, max(Y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (0, 0, 255),
            2
        )


def show_video(path, width=700):
    mp4 = open(path, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

    return HTML(f"""
    <video width="{width}" controls>
        <source src="{data_url}" type="video/mp4">
    </video>
    """)


# ============================================================
# 4. Main Processing Loop
# ============================================================

frame_idx = 0
start_time = time.time()
processed_frames = 0
fps_list = []

while True:
    ret, frame_full = cap.read()
    if not ret:
        break

    result_frame = frame_full.copy()

    frame_small = cv2.resize(frame_full, (process_w, process_h))
    gray = cv2.cvtColor(frame_small, cv2.COLOR_BGR2GRAY)

    # ========================================================
    # 是否在這一幀執行完整偵測
    # ========================================================

    do_detection = (frame_idx % detection_interval == 0)

    if do_detection:
        # 每次完整偵測前，清空上一輪結果
        last_fire_boxes = []

        # ----------------------------------------------------
        # Step 1: HSV candidate mask
        # ----------------------------------------------------
        fire_mask = build_hsv_fire_mask(frame_small)
        hsv_vis_small = cv2.cvtColor(fire_mask, cv2.COLOR_GRAY2BGR)

        flow_debug_small = np.zeros((process_h, process_w), dtype=np.uint8)

        # ----------------------------------------------------
        # Step 2: Find HSV candidate contours
        # ----------------------------------------------------
        contours, _ = cv2.findContours(
            fire_mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        # ----------------------------------------------------
        # Step 3: Run Dense Optical Flow only inside each ROI
        # ----------------------------------------------------
        for cnt in contours:
            area = cv2.contourArea(cnt)

            if area < min_area:
                continue

            if area > max_area_ratio * process_w * process_h:
                continue

            x, y, bw, bh = cv2.boundingRect(cnt)

            x1 = max(x - roi_pad, 0)
            y1 = max(y - roi_pad, 0)
            x2 = min(x + bw + roi_pad, process_w)
            y2 = min(y + bh + roi_pad, process_h)

            roi_w = x2 - x1
            roi_h = y2 - y1

            if roi_w < min_roi_w or roi_h < min_roi_h:
                continue

            prev_roi = prev_gray[y1:y2, x1:x2]
            gray_roi = gray[y1:y2, x1:x2]
            roi_mask = fire_mask[y1:y2, x1:x2]

            roi_mag, roi_ang = compute_dense_flow_roi(prev_roi, gray_roi)

            is_fire, mean_mag, motion_density, score = evaluate_fire_region(
                roi_mask,
                roi_mag
            )

            roi_flow_mask = (roi_mag > flow_mag_threshold).astype(np.uint8) * 255
            roi_flow_mask = cv2.morphologyEx(roi_flow_mask, cv2.MORPH_OPEN, kernel)

            flow_debug_small[y1:y2, x1:x2] = cv2.bitwise_or(
                flow_debug_small[y1:y2, x1:x2],
                roi_flow_mask
            )

            if is_fire:
                X1 = int(x1 * scale_x)
                Y1 = int(y1 * scale_y)
                X2 = int(x2 * scale_x)
                Y2 = int(y2 * scale_y)

                last_fire_boxes.append({
                    "box": (X1, Y1, X2, Y2),
                    "mean_mag": mean_mag,
                    "motion_density": motion_density
                })

                cv2.rectangle(
                    hsv_vis_small,
                    (x1, y1),
                    (x2, y2),
                    (0, 0, 255),
                    1
                )

                cv2.rectangle(
                    flow_debug_small,
                    (x1, y1),
                    (x2, y2),
                    255,
                    1
                )

        # 更新 debug 影片畫面
        last_hsv_vis_small = hsv_vis_small.copy()
        last_flow_vis_small = cv2.cvtColor(flow_debug_small, cv2.COLOR_GRAY2BGR)

    else:
        # 非偵測幀：沿用上一輪 HSV / flow debug 結果
        hsv_vis_small = last_hsv_vis_small.copy()
        flow_in_hsv_vis_small = last_flow_vis_small.copy()

    # ========================================================
    # 不論是否偵測，都畫上一輪的 fire boxes
    # ========================================================

    draw_last_boxes(result_frame, last_fire_boxes)

    # 標記目前是否有跑 detection，方便 debug
    mode_text = "DETECT" if do_detection else "TRACK-HOLD"

    cv2.putText(
        result_frame,
        mode_text,
        (20, 75),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 0),
        2
    )

    # ========================================================
    # FPS calculation
    # ========================================================

    processed_frames += 1
    elapsed_time = time.time() - start_time
    processing_fps = processed_frames / elapsed_time
    fps_list.append(processing_fps)

    cv2.putText(
        result_frame,
        f"FPS: {processing_fps:.2f}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

    # ========================================================
    # Resize debug videos back to original size
    # ========================================================

    hsv_vis_full = cv2.resize(
        hsv_vis_small,
        (orig_w, orig_h),
        interpolation=cv2.INTER_NEAREST
    )

    flow_vis_full = cv2.resize(
        last_flow_vis_small,
        (orig_w, orig_h),
        interpolation=cv2.INTER_NEAREST
    )

    # ========================================================
    # Write videos
    # ========================================================

    out_fire.write(result_frame)
    out_hsv.write(hsv_vis_full)
    out_flow.write(flow_vis_full)

    # 注意：
    # prev_gray 仍然每一幀更新，這樣下一次 detection 的 flow 是跟前一幀比較
    # 如果你想比較 detection frame 間隔的變化，可以改成只有 do_detection 時才更新
    prev_gray = gray.copy()

    frame_idx += 1

cap.release()
out_fire.release()
out_hsv.release()
out_flow.release()

avg_fps = processed_frames / (time.time() - start_time)

print("Done.")
print(f"Original resolution: {orig_w} x {orig_h}")
print(f"Processing resolution: {process_w} x {process_h}")
print(f"Detection interval: every {detection_interval} frames")
print(f"Processed frames: {processed_frames}")
print(f"Average Processing FPS: {avg_fps:.2f}")
print(f"Input Video FPS: {fps:.2f}")
print("Fire detection output:", out_fire_path)
print("HSV mask output:", out_hsv_path)
print("Dense flow mask output:", out_flow_path)

Done.
Original resolution: 1280 x 720
Processing resolution: 640 x 480
Detection interval: every 2 frames
Processed frames: 320
Average Processing FPS: 29.16
Input Video FPS: 30.00
Fire detection output: /content/out_fire_detection_roi_flow_skip.mp4
HSV mask output: /content/out_hsv_mask_roi_flow_skip.mp4
Dense flow mask output: /content/out_dense_flow_mask_roi_flow_skip.mp4


# HSV -> ROI -> farneback (Frame skipping and missing prevention)

In [ ]:
import time
import cv2
import numpy as np
from base64 import b64encode
from IPython.display import HTML

# ============================================================
# 1. Input / Output Settings
# ============================================================

video_path = "fire.mp4"

out_fire_path = "/content/out_fire_detection_roi_flow_skip.mp4"
out_hsv_path = "/content/out_hsv_mask_roi_flow_skip.mp4"
out_flow_path = "/content/out_dense_flow_mask_roi_flow_skip.mp4"

cap = cv2.VideoCapture(video_path)

ret, prev_frame_full = cap.read()
if not ret:
    raise RuntimeError("Cannot read input video. Please check video_path.")

fps = cap.get(cv2.CAP_PROP_FPS)
orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if fps == 0 or np.isnan(fps):
    fps = 30

# ============================================================
# 1.1 Processing Resolution
# ============================================================

process_w = 640
process_h = 480

scale_x = orig_w / process_w
scale_y = orig_h / process_h

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out_fire = cv2.VideoWriter(out_fire_path, fourcc, fps, (orig_w, orig_h))
out_hsv = cv2.VideoWriter(out_hsv_path, fourcc, fps, (orig_w, orig_h))
out_flow = cv2.VideoWriter(out_flow_path, fourcc, fps, (orig_w, orig_h))

prev_frame_small = cv2.resize(prev_frame_full, (process_w, process_h))
prev_gray = cv2.cvtColor(prev_frame_small, cv2.COLOR_BGR2GRAY)

# ============================================================
# 2. Parameters
# ============================================================

# ---------- Detection interval ----------
# 每 N 幀才跑一次完整偵測，其餘幀沿用上一輪 bounding boxes
detection_interval = 3

# ---------- Miss tolerance ----------
# 連續幾次完整 detection 沒抓到火焰，才清空框
max_missed_detections = 3
missed_detection_count = 0

# 儲存上一輪有效偵測結果
last_fire_boxes = []
last_hsv_vis_small = np.zeros((process_h, process_w, 3), dtype=np.uint8)
last_flow_vis_small = np.zeros((process_h, process_w, 3), dtype=np.uint8)

# ---------- HSV fire candidate parameters ----------

# 目前你的版本：範圍較寬，適合白亮火焰與偏亮區域
lower_fire_color = np.array([0, 0, 200])
upper_fire_color = np.array([90, 255, 255])

# 白亮火焰核心
lower_white_fire = np.array([0, 0, 230])
upper_white_fire = np.array([179, 80, 255])

# 淡黃色高亮區域
lower_pale_yellow = np.array([10, 20, 200])
upper_pale_yellow = np.array([45, 120, 255])

# ---------- Morphology ----------
kernel = np.ones((5, 5), np.uint8)

# ---------- Contour filtering ----------
min_area_original = 100
min_area = max(20, int(min_area_original / (scale_x * scale_y)))

max_area_ratio = 0.7

roi_pad = 5
min_roi_w = 10
min_roi_h = 10

# ---------- Dense optical flow parameters ----------
flow_mag_threshold = 1.2
mean_mag_threshold = 0.6
motion_density_threshold = 0.12

# ---------- Fire decision ----------
min_fire_score = 2

# ============================================================
# 3. Helper Functions
# ============================================================

def build_hsv_fire_mask(frame_small):
    hsv = cv2.cvtColor(frame_small, cv2.COLOR_BGR2HSV)

    mask_color = cv2.inRange(hsv, lower_fire_color, upper_fire_color)
    mask_white = cv2.inRange(hsv, lower_white_fire, upper_white_fire)
    mask_pale_yellow = cv2.inRange(hsv, lower_pale_yellow, upper_pale_yellow)

    # 目前使用你的設定：只使用 mask_color
    # 若之後想合併白亮核心與淡黃色區域，可改回 bitwise_or 版本
    fire_mask = mask_color

    # 合併版本可用下面三行取代 fire_mask = mask_color
    # fire_mask = cv2.bitwise_or(mask_color, mask_white)
    # fire_mask = cv2.bitwise_or(fire_mask, mask_pale_yellow)

    fire_mask = cv2.morphologyEx(fire_mask, cv2.MORPH_OPEN, kernel)
    fire_mask = cv2.morphologyEx(fire_mask, cv2.MORPH_CLOSE, kernel)

    return fire_mask


def compute_dense_flow_roi(prev_gray_roi, gray_roi):
    flow = cv2.calcOpticalFlowFarneback(
        prev_gray_roi,
        gray_roi,
        None,
        pyr_scale=0.5,
        levels=1,
        winsize=9,
        iterations=1,
        poly_n=5,
        poly_sigma=1.1,
        flags=0
    )

    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    return mag, ang


def evaluate_fire_region(roi_mask, roi_mag):
    valid_pixels = roi_mask > 0

    if np.sum(valid_pixels) == 0:
        return False, 0.0, 0.0, 0

    roi_motion = roi_mag[valid_pixels]

    mean_mag = float(np.mean(roi_motion))
    motion_density = float(np.sum(roi_motion > flow_mag_threshold) / len(roi_motion))

    score = 0

    if mean_mag >= mean_mag_threshold:
        score += 1

    if motion_density >= motion_density_threshold:
        score += 1

    is_fire = score >= min_fire_score

    return is_fire, mean_mag, motion_density, score


def draw_last_boxes(result_frame, boxes):
    """
    在非偵測幀，或 missed tolerance 尚未達門檻時，
    沿用上一輪有效 bounding boxes。
    """

    for item in boxes:
        X1, Y1, X2, Y2 = item["box"]
        mean_mag = item["mean_mag"]
        motion_density = item["motion_density"]

        cv2.rectangle(
            result_frame,
            (X1, Y1),
            (X2, Y2),
            (0, 0, 255),
            2
        )

        label = f"FIRE M={mean_mag:.2f} D={motion_density:.2f}"

        cv2.putText(
            result_frame,
            label,
            (X1, max(Y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (0, 0, 255),
            2
        )


def show_video(path, width=700):
    mp4 = open(path, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

    return HTML(f"""
    <video width="{width}" controls>
        <source src="{data_url}" type="video/mp4">
    </video>
    """)


# ============================================================
# 4. Main Processing Loop
# ============================================================

frame_idx = 0
start_time = time.time()
processed_frames = 0
fps_list = []

while True:
    ret, frame_full = cap.read()
    if not ret:
        break

    result_frame = frame_full.copy()

    frame_small = cv2.resize(frame_full, (process_w, process_h))
    gray = cv2.cvtColor(frame_small, cv2.COLOR_BGR2GRAY)

    # ========================================================
    # 是否在這一幀執行完整偵測
    # ========================================================

    do_detection = (frame_idx % detection_interval == 0)

    if do_detection:
        # 本次 detection 的暫存結果
        # 注意：這裡不要清空 last_fire_boxes
        current_fire_boxes = []

        # ----------------------------------------------------
        # Step 1: HSV candidate mask
        # ----------------------------------------------------

        fire_mask = build_hsv_fire_mask(frame_small)
        hsv_vis_small = cv2.cvtColor(fire_mask, cv2.COLOR_GRAY2BGR)

        flow_debug_small = np.zeros((process_h, process_w), dtype=np.uint8)

        # ----------------------------------------------------
        # Step 2: Find HSV candidate contours
        # ----------------------------------------------------

        contours, _ = cv2.findContours(
            fire_mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        # ----------------------------------------------------
        # Step 3: Run Dense Optical Flow only inside each ROI
        # ----------------------------------------------------

        for cnt in contours:
            area = cv2.contourArea(cnt)

            if area < min_area:
                continue

            if area > max_area_ratio * process_w * process_h:
                continue

            x, y, bw, bh = cv2.boundingRect(cnt)

            x1 = max(x - roi_pad, 0)
            y1 = max(y - roi_pad, 0)
            x2 = min(x + bw + roi_pad, process_w)
            y2 = min(y + bh + roi_pad, process_h)

            roi_w = x2 - x1
            roi_h = y2 - y1

            if roi_w < min_roi_w or roi_h < min_roi_h:
                continue

            prev_roi = prev_gray[y1:y2, x1:x2]
            gray_roi = gray[y1:y2, x1:x2]
            roi_mask = fire_mask[y1:y2, x1:x2]

            roi_mag, roi_ang = compute_dense_flow_roi(prev_roi, gray_roi)

            is_fire, mean_mag, motion_density, score = evaluate_fire_region(
                roi_mask,
                roi_mag
            )

            roi_flow_mask = (roi_mag > flow_mag_threshold).astype(np.uint8) * 255
            roi_flow_mask = cv2.morphologyEx(roi_flow_mask, cv2.MORPH_OPEN, kernel)

            flow_debug_small[y1:y2, x1:x2] = cv2.bitwise_or(
                flow_debug_small[y1:y2, x1:x2],
                roi_flow_mask
            )

            if is_fire:
                X1 = int(x1 * scale_x)
                Y1 = int(y1 * scale_y)
                X2 = int(x2 * scale_x)
                Y2 = int(y2 * scale_y)

                current_fire_boxes.append({
                    "box": (X1, Y1, X2, Y2),
                    "mean_mag": mean_mag,
                    "motion_density": motion_density
                })

                cv2.rectangle(
                    hsv_vis_small,
                    (x1, y1),
                    (x2, y2),
                    (0, 0, 255),
                    1
                )

                cv2.rectangle(
                    flow_debug_small,
                    (x1, y1),
                    (x2, y2),
                    255,
                    1
                )

        # ----------------------------------------------------
        # Step 4: Miss tolerance update
        # ----------------------------------------------------

        if len(current_fire_boxes) > 0:
            # 本次偵測有抓到火焰：
            # 更新有效框，miss 歸零
            last_fire_boxes = current_fire_boxes
            missed_detection_count = 0
            detection_status = "FIRE-DETECTED"

        else:
            # 本次完整 detection 沒抓到火焰：
            # 不立即清空框，而是累計 miss 次數
            missed_detection_count += 1

            if missed_detection_count >= max_missed_detections:
                # 連續 3 次完整 detection 都沒抓到，才判斷無火並清空框
                last_fire_boxes = []
                detection_status = "NO-FIRE"
            else:
                # 尚未達門檻，保留上一輪有效框
                detection_status = "MISS-HOLD"

        # ----------------------------------------------------
        # Step 5: Update debug videos
        # ----------------------------------------------------

        last_hsv_vis_small = hsv_vis_small.copy()
        last_flow_vis_small = cv2.cvtColor(flow_debug_small, cv2.COLOR_GRAY2BGR)

    else:
        # 非偵測幀：沿用上一輪 HSV / flow debug 結果
        hsv_vis_small = last_hsv_vis_small.copy()
        detection_status = "HOLD"

    # ========================================================
    # 不論是否偵測，都畫 last_fire_boxes
    # 只有連續 miss 達門檻後，last_fire_boxes 才會被清空
    # ========================================================

    draw_last_boxes(result_frame, last_fire_boxes)

    # ========================================================
    # Display mode / status
    # ========================================================

    if do_detection:
        mode_text = f"DETECT | {detection_status} | miss={missed_detection_count}/{max_missed_detections}"
    else:
        mode_text = f"HOLD | miss={missed_detection_count}/{max_missed_detections}"

    cv2.putText(
        result_frame,
        mode_text,
        (20, 75),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 0),
        2
    )

    # ========================================================
    # FPS calculation
    # ========================================================

    processed_frames += 1
    elapsed_time = time.time() - start_time
    processing_fps = processed_frames / elapsed_time
    fps_list.append(processing_fps)

    cv2.putText(
        result_frame,
        f"FPS: {processing_fps:.2f}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

    # ========================================================
    # Resize debug videos back to original size
    # ========================================================

    hsv_vis_full = cv2.resize(
        hsv_vis_small,
        (orig_w, orig_h),
        interpolation=cv2.INTER_NEAREST
    )

    flow_vis_full = cv2.resize(
        last_flow_vis_small,
        (orig_w, orig_h),
        interpolation=cv2.INTER_NEAREST
    )

    # ========================================================
    # Write videos
    # ========================================================

    out_fire.write(result_frame)
    out_hsv.write(hsv_vis_full)
    out_flow.write(flow_vis_full)

    # 每一幀都更新 prev_gray
    # 下一次 detection 時，光流會比較 detection 當下與前一幀的變化
    prev_gray = gray.copy()

    frame_idx += 1

cap.release()
out_fire.release()
out_hsv.release()
out_flow.release()

avg_fps = processed_frames / (time.time() - start_time)

print("Done.")
print(f"Original resolution: {orig_w} x {orig_h}")
print(f"Processing resolution: {process_w} x {process_h}")
print(f"Detection interval: every {detection_interval} frames")
print(f"Max missed detections: {max_missed_detections}")
print(f"Processed frames: {processed_frames}")
print(f"Average Processing FPS: {avg_fps:.2f}")
print(f"Input Video FPS: {fps:.2f}")
print("Fire detection output:", out_fire_path)
print("HSV mask output:", out_hsv_path)
print("Dense flow mask output:", out_flow_path)

Done.
Original resolution: 1280 x 720
Processing resolution: 640 x 480
Detection interval: every 3 frames
Max missed detections: 3
Processed frames: 320
Average Processing FPS: 37.17
Input Video FPS: 30.00
Fire detection output: /content/out_fire_detection_roi_flow_skip.mp4
HSV mask output: /content/out_hsv_mask_roi_flow_skip.mp4
Dense flow mask output: /content/out_dense_flow_mask_roi_flow_skip.mp4
